# 지속 성장 카테고리 발굴 + 블랙프라이데이 전략

**주제**  
1. **2016–2018 브라질 Google Trends**를 보고, **지속적으로 성장 중인 제품 카테고리**를 발굴 (단, Olist에서 실제 판매되는 **주요 카테고리**만 대상).  
2. **블랙프라이데이** 등 특정 이벤트 시기를 중점으로 Olist 판매 데이터를 분석.  
3. 이를 바탕으로 **다음 블랙프라이데이 전략 방향** 제시.

**전제**  
- Google Trends 데이터: `02-Google_Trends_BR_2016-2018.ipynb` 실행 후 생성된 `google_trends_br_2016_2018_olist.csv` 사용 (없으면 해당 노트북 먼저 실행).  
- 블랙프라이데이: 브라질 기준 **11월 마지막 금요일**이 있는 주를 "BF 주"로 정의.

---
## 1. 데이터 로드

**흐름**: Olist 주문·상품·카테고리 로드 → 주문 월/주 단위 컬럼 추가. Google Trends CSV 로드.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Olist 데이터
DATA_DIR = Path.cwd() / "data"
for _ in [Path.cwd() / "data", Path.cwd().parent / "예측 대시보드 용 프로젝트" / "data"]:
    if (_ / "orders_delivered.csv").exists():
        DATA_DIR = _
        break

if not (DATA_DIR / "orders_delivered.csv").exists():
    import kagglehub
    _path = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    orders = pd.read_csv(_path / "olist_orders_dataset.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
    orders = orders[orders["order_status"] == "delivered"].copy()
    order_items = pd.read_csv(_path / "olist_order_items_dataset.csv")
    products = pd.read_csv(_path / "olist_products_dataset.csv")
    print("(data/ 없음 → kagglehub에서 로드)")
else:
    orders = pd.read_csv(DATA_DIR / "orders_delivered.csv")
    order_items = pd.read_csv(DATA_DIR / "order_items.csv")
    products = pd.read_csv(DATA_DIR / "products.csv")
    orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")

products["product_category_name"] = products["product_category_name"].fillna("unknown")
ord = order_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
ord["product_category_name"] = ord["product_category_name"].fillna("unknown")
ord = ord.merge(orders[["order_id", "order_purchase_timestamp"]], on="order_id")
ord["order_month"] = ord["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
ord["order_week_start"] = ord["order_purchase_timestamp"].dt.to_period("W-MON").dt.start_time

# Google Trends CSV (02번 노트북 실행 후 생성)
TRENDS_CSV = Path.cwd() / "google_trends_br_2016_2018_olist.csv"
for _ in [Path.cwd(), Path.cwd().parent / "예측 대시보드 용 프로젝트"]:
    if (_ / "google_trends_br_2016_2018_olist.csv").exists():
        TRENDS_CSV = _ / "google_trends_br_2016_2018_olist.csv"
        break

if TRENDS_CSV.exists():
    trends = pd.read_csv(TRENDS_CSV)
    trends["date"] = pd.to_datetime(trends["date"], errors="coerce")
    trends = trends.dropna(subset=["date"])
    print("Trends 행:", len(trends), "| 키워드 수:", trends["keyword"].nunique() if "keyword" in trends.columns else "-")
else:
    trends = pd.DataFrame()
    print("Google Trends CSV 없음. 02-Google_Trends_BR_2016-2018.ipynb 실행 후 google_trends_br_2016_2018_olist.csv 생성 필요.")

print("Olist ord 행:", len(ord), "| 기간:", ord["order_purchase_timestamp"].min().date(), "~", ord["order_purchase_timestamp"].max().date())

---
## 2. Olist 주요 판매 카테고리 (Trends와 매칭용)

**흐름**: 판매량 상위 N개 카테고리만 사용. 키워드 = 카테고리명 언더스코어→공백 (Trends 키워드와 동일 형식).

In [ ]:
# 판매량(주문 건수) 상위 15개 카테고리 = Olist "주요" 카테고리
top_cats = ord.groupby("product_category_name")["order_id"].nunique().sort_values(ascending=False)
top_cats = top_cats[top_cats.index != "unknown"].head(15)
olist_main_categories = top_cats.index.tolist()
# Trends 키워드 형식: 언더스코어 → 공백
keyword_to_cat = {c.replace("_", " "): c for c in olist_main_categories}
cat_to_keyword = {c: c.replace("_", " ") for c in olist_main_categories}

print("Olist 주요 카테고리(상위 15):", olist_main_categories)
print("키워드 매핑 샘플:", list(cat_to_keyword.items())[:5])

---
## 3. 지속 성장 카테고리 발굴 (Google Trends)

**흐름**: 연도별(2016, 2017, 2018) 평균 검색 관심도(interest) 계산 → 2018 > 2016 이고 2017 대비 2018 성장이 있으면 "지속 성장"으로 분류. (Trends 데이터가 없으면 Olist만으로 성장 추정: 연도별 매출 성장률 사용.)

In [ ]:
if not trends.empty and "keyword" in trends.columns and "interest" in trends.columns:
    trends["year"] = trends["date"].dt.year
    # 연도별 키워드 평균 관심도 (0~100 상대 지수)
    yearly = trends.groupby(["keyword", "year"])["interest"].mean().reset_index()
    yearly_wide = yearly.pivot(index="keyword", columns="year", values="interest").reset_index()
    for y in [2016, 2017, 2018]:
        if y not in yearly_wide.columns:
            yearly_wide[y] = np.nan
    yearly_wide = yearly_wide.rename(columns={2016: "avg_2016", 2017: "avg_2017", 2018: "avg_2018"})
    # Olist 주요 카테고리 키워드만 (매칭되는 것)
    keywords_olist = list(keyword_to_cat.keys())
    yearly_olist = yearly_wide[yearly_wide["keyword"].isin(keywords_olist)].copy()
    yearly_olist["growth_16_18"] = yearly_olist["avg_2018"] - yearly_olist["avg_2016"]
    yearly_olist["growth_17_18"] = yearly_olist["avg_2018"] - yearly_olist["avg_2017"]
    yearly_olist["sustained_growth"] = (yearly_olist["avg_2018"] > yearly_olist["avg_2016"]) & (yearly_olist["growth_16_18"] > 0)
    yearly_olist["product_category_name"] = yearly_olist["keyword"].map(keyword_to_cat)
    growing_categories = yearly_olist[yearly_olist["sustained_growth"]].sort_values("growth_16_18", ascending=False)
    print("Trends 기반 지속 성장 카테고리 (2018 > 2016):")
    print(growing_categories[["keyword", "avg_2016", "avg_2017", "avg_2018", "growth_16_18"]].to_string(index=False))
    growing_cat_list = growing_categories["product_category_name"].dropna().unique().tolist()
else:
    # Trends 없을 때: Olist 연도별 매출 성장으로 대체
    ord["year"] = ord["order_purchase_timestamp"].dt.year
    rev_by_cat_year = ord.groupby(["product_category_name", "year"])["price"].sum().reset_index()
    rev_wide = rev_by_cat_year.pivot(index="product_category_name", columns="year", values="price").reset_index()
    for y in [2016, 2017, 2018]:
        if y not in rev_wide.columns:
            rev_wide[y] = 0
    rev_wide["growth_16_18"] = rev_wide[2018] - rev_wide[2016]
    rev_wide = rev_wide[rev_wide["product_category_name"].isin(olist_main_categories)]
    rev_wide["sustained_growth"] = rev_wide[2018] > rev_wide[2016]
    growing_categories = rev_wide[rev_wide["sustained_growth"]].sort_values("growth_16_18", ascending=False)
    print("Trends 없음 → Olist 매출 성장(2018 vs 2016)으로 대체:")
    print(growing_categories[["product_category_name", 2016, 2017, 2018, "growth_16_18"]].head(15).to_string(index=False))
    growing_cat_list = growing_categories["product_category_name"].tolist()

print("\n지속 성장 카테고리 목록:", growing_cat_list[:10])

---
## 4. 블랙프라이데이 주(BF week) 정의

**흐름**: 브라질 블랙프라이데이 = 11월 마지막 금요일. 해당 주(월요일 시작)를 `bf_week_start`로 두고, 주문에 `is_bf_week` 플래그 부여.

In [ ]:
# 11월 마지막 금요일이 속한 주의 월요일 = BF week 시작일
def get_bf_week_start(year):
    # 11월 마지막 날
    end_nov = pd.Timestamp(year=year, month=11, day=30)
    # 마지막 금요일: 30에서 거꾸로
    for d in range(30, 0, -1):
        t = pd.Timestamp(year=year, month=11, day=d)
        if t.dayofweek == 4:  # Friday
            break
    # 해당 주 월요일
    friday = t
    monday = friday - pd.Timedelta(days=4)
    return monday

bf_weeks = [get_bf_week_start(y) for y in [2016, 2017, 2018]]
print("블랙프라이데이 주(월요일):", bf_weeks)

ord["is_bf_week"] = ord["order_week_start"].isin(bf_weeks)
ord["is_november"] = ord["order_purchase_timestamp"].dt.month == 11
bf_orders = ord[ord["is_bf_week"]].copy()
print("BF 주 주문 건수:", bf_orders["order_id"].nunique(), "| BF 주 매출:", bf_orders["price"].sum().round(0))

---
## 5. 블랙프라이데이 vs 비-BF 시기 비교 (카테고리별)

**흐름**: BF 주 vs 11월 나머지 주(또는 10월) 카테고리별 매출·주문 수 비교 → BF에서 특히 잘 나온 카테고리, BF 대비 평소 대비 배수 계산.

In [ ]:
# 11월 전체 중 BF 주 vs 11월 비-BF 주 비교
nov = ord[ord["is_november"]].copy()
nov_bf = nov[nov["is_bf_week"]].groupby("product_category_name").agg(
    bf_revenue=("price", "sum"), bf_orders=("order_id", "nunique"), bf_items=("order_id", "count"),
).reset_index()
nov_non_bf = nov[~nov["is_bf_week"]].groupby("product_category_name").agg(
    non_bf_revenue=("price", "sum"), non_bf_orders=("order_id", "nunique"), non_bf_items=("order_id", "count"),
).reset_index()

bf_compare = nov_bf.merge(nov_non_bf, on="product_category_name", how="outer").fillna(0)
bf_compare["revenue_lift"] = (bf_compare["bf_revenue"] / (bf_compare["non_bf_revenue"] + 1)).round(2)
bf_compare["order_lift"] = (bf_compare["bf_orders"] / (bf_compare["non_bf_orders"] + 1)).round(2)
bf_compare = bf_compare.sort_values("bf_revenue", ascending=False)

print("11월: BF 주 vs 11월 비-BF 주 — 카테고리별 매출·주문·배수(lift):")
print(bf_compare.head(15).to_string(index=False))

# BF에서 특히 반응 좋은 카테고리 (lift > 1.5 또는 BF 매출 상위)
bf_responsive = bf_compare[(bf_compare["revenue_lift"] >= 1.2) | (bf_compare["bf_revenue"] >= bf_compare["bf_revenue"].quantile(0.7))].sort_values("bf_revenue", ascending=False)
print("\nBF 반응 우수 카테고리 (lift>=1.2 또는 BF 매출 상위 30%):")
print(bf_responsive[["product_category_name", "bf_revenue", "non_bf_revenue", "revenue_lift"]].head(10).to_string(index=False))

---
## 6. 연도별 블랙프라이데이 추이

**흐름**: 2016·2017·2018 각 BF 주의 카테고리별 매출·주문 수 → 전년 대비 성장률.

In [ ]:
bf_orders["year"] = bf_orders["order_purchase_timestamp"].dt.year
bf_by_year = bf_orders.groupby(["year", "product_category_name"]).agg(
    revenue=("price", "sum"), orders=("order_id", "nunique"),
).reset_index()

yearly_bf_totals = bf_orders.groupby("year").agg(
    total_revenue=("price", "sum"), total_orders=("order_id", "nunique"),
).reset_index()
print("연도별 BF 주 전체 매출·주문:")
print(yearly_bf_totals.to_string(index=False))

pivot_bf = bf_by_year.pivot(index="product_category_name", columns="year", values="revenue").fillna(0)
if 2017 in pivot_bf.columns and 2016 in pivot_bf.columns:
    pivot_bf["growth_16_17"] = (pivot_bf[2017] - pivot_bf[2016]) / (pivot_bf[2016] + 1) * 100
if 2018 in pivot_bf.columns and 2017 in pivot_bf.columns:
    pivot_bf["growth_17_18"] = (pivot_bf[2018] - pivot_bf[2017]) / (pivot_bf[2017] + 1) * 100
pivot_bf = pivot_bf.sort_values(2018 if 2018 in pivot_bf.columns else 2017, ascending=False)
print("\n카테고리별 BF 주 매출 연도 추이 (상위 10):")
print(pivot_bf.head(10).to_string())

---
## 7. 다음 블랙프라이데이 전략 방향 제시

**흐름**: (1) 지속 성장 카테고리 + (2) BF 반응 우수 카테고리 교집합/합집합 → 우선 집중 카테고리. (3) 연도별 BF 성장 추이 반영. (4) 전략 요약 표·문장으로 정리.

In [ ]:
# 지속 성장이면서 BF에서도 잘 나온 카테고리 = 다음 BF 핵심 집중 후보
growth_set = set(growing_cat_list[:10]) if growing_cat_list else set(olist_main_categories[:5])
bf_top_set = set(bf_responsive["product_category_name"].head(10).tolist())
priority_categories = list(growth_set | bf_top_set)
priority_both = list(growth_set & bf_top_set)

print("【다음 블랙프라이데이 전략 방향】")
print("=" * 50)
print("1. 지속 성장 카테고리 (Trends/Olist 성장):", growing_cat_list[:8])
print("2. BF 반응 우수 카테고리:", bf_responsive["product_category_name"].head(8).tolist())
print("3. 두 조건 모두 만족(우선 집중):", priority_both if priority_both else "(교집합 없음 → 성장 또는 BF 반응 중 하나라도 만족한 카테고리 활용)")
print("4. 전략 우선순위 카테고리(합집합):", priority_categories[:12])
print("=" * 50)

---
## 8. 결론 및 전략 요약

### 8.1 분석 요약

| 단계 | 내용 |
|------|------|
| 지속 성장 카테고리 | Google Trends 2016 vs 2018 (또는 Olist 매출 연도별)로 검색 관심도·매출이 **꾸준히 증가**한 Olist 주요 카테고리 도출 |
| BF 시기 정의 | 11월 마지막 금요일이 포함된 주를 "BF 주"로 두고, 해당 주 vs 11월 비-BF 주 비교 |
| BF 반응 우수 | 카테고리별 BF 주 매출·주문 수와 **revenue_lift**(BF/비-BF 배수)로 BF에 반응이 큰 카테고리 식별 |
| 연도별 BF 추이 | 2016·2017·2018 BF 주 매출·주문·카테고리별 성장률로 트렌드 확인 |

### 8.2 다음 블랙프라이데이 전략 방향 (제안)

1. **재고·프로모션 집중**  
   - **지속 성장 + BF 반응 우수** 카테고리를 최우선으로 재고 확보·할인/쿠폰 설계.  
   - 지속 성장만 있는 카테고리는 "성장 테스트" 차원에서 BF에서 노출·소규모 프로모션으로 반응 측정.  

2. **가격·할인 전략**  
   - 과거 BF에서 **revenue_lift**가 높았던 카테고리는 할인 깊이를 유지하거나 소폭 확대 검토.  
   - 연도별 BF 매출이 크게 성장한 카테고리는 경쟁 심화 가능성이 있으므로, 가격 대비 품질·배송 속도 차별화.  

3. **마케팅·노출**  
   - BF 주 전 1~2주부터 "지속 성장 카테고리" 키워드로 검색/디스플레이 광고 강화.  
   - 이메일·푸시: 과거 BF 구매 고객에게 **BF 반응 우수 카테고리** 위주 추천.  

4. **배송·운영**  
   - BF 주 주문 집중에 대비해 상위 카테고리 재고를 물리적으로 가까운 창고에 배치, 배송일 예측·고객 안내 강화.  

5. **모니터링 지표**  
   - 다음 BF 후: 카테고리별 **BF vs 비-BF revenue_lift**, 연도 대비 BF 매출 성장률, 지속 성장 카테고리 실제 매출 점유율을 추적해 전략 효과 검증.